# Pangolin — splice-site prediction & variant scoring

![Pangolin — splice-site prediction & variant scoring](https://proto-bio.github.io/proto-assets/images/tool/pangolin/hero.png)

Pangolin is a SpliceAI-lineage deep-learning model that predicts tissue-specific splice-site probability from raw DNA and scores the splicing effect (gain/loss) of variants. This notebook demonstrates both registered tools: `run_pangolin_predict` for per-position splice-probability prediction and `run_pangolin_score_variants` for variant splice-effect scoring.

- Paper: https://doi.org/10.1186/s13059-022-02664-4
- Repository: https://github.com/tkzeng/Pangolin

In [ ]:
# Public API for both Pangolin tools, plus the notebook docs helper.
from proto_tools.tools.rna_splicing.pangolin import (
    run_pangolin_predict,
    PangolinPredictInput,
    PangolinPredictConfig,
    run_pangolin_score_variants,
    PangolinScoreVariantsInput,
    PangolinScoreVariantsConfig,
    PangolinVariant,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [ ]:
# Input schema for splice-site prediction: the DNA sequences to score.
display_api_reference("pangolin-predict", "input", "run_pangolin_predict")

In [ ]:
# Config schema: which tissues to ensemble and which device to run on.
display_api_reference("pangolin-predict", "config", "run_pangolin_predict")

In [ ]:
# Output schema: per-sequence predictions with per-position, per-tissue scores.
display_api_reference("pangolin-predict", "output", "run_pangolin_predict")

## Splice-site prediction

Pangolin needs at least 10,001 bp of input: it uses 5,000 bp of flanking context on each side and scores the central `len(sequence) - 10000` positions. We use the real 10,106 bp HBB genomic window bundled next to this notebook.

In [ ]:
from pathlib import Path

# Real HBB (hemoglobin beta) genomic window. Pangolin scores the central positions
# (5,000 bp of flank on each side), which span the HBB splice sites.
hbb = Path("hbb_genomic.fasta").read_text()
seq = "".join(line for line in hbb.splitlines() if not line.startswith(">"))

result = run_pangolin_predict(
    PangolinPredictInput(sequences=[seq]),
    PangolinPredictConfig(device="cpu"),
)

prediction = result.results[0]
print(f"success: {result.success}")
print(f"scored positions: {len(prediction.scores)}")
print(f"tissues: {prediction.tissues}")
# Strongest splice signal across the scored window (max |score| over tissues).
import numpy as np
arr = np.abs(np.array(prediction.scores))
best = int(arr.max(axis=1).argmax())
print(f"max splice score {arr[best].max():.3f} at scored index {best}")

In [ ]:
# Advanced: restrict the ensemble to a subset of tissues. Each score row then
# has one column per requested tissue, in the order given.
subset = run_pangolin_predict(
    PangolinPredictInput(sequences=[seq]),
    PangolinPredictConfig(device="cpu", tissues=["BRAIN", "HEART"]),
)

subset_prediction = subset.results[0]
print(f"tissues: {subset_prediction.tissues}")
print(f"score shape: {len(subset_prediction.scores)} positions x "
      f"{len(subset_prediction.scores[0])} tissues")

## Variant splice-effect scoring

`run_pangolin_score_variants` compares the predicted splice-site probability between the reference and alternate sequence over a `± distance` window, reducing across the selected tissues to a per-position splice gain (largest increase) and loss (largest decrease). Each variant carries its own reference window — no genome FASTA is required.

In [ ]:
# Place a single-nucleotide variant at the HBB splice site found above (position 5032),
# reusing the HBB window (Pangolin needs >= 5,000 bp of flank on each side).
pos = 5032
ref = seq[pos]
alt = "A" if ref != "A" else "G"

variant = PangolinVariant(
    sequence=seq,
    variant_position=pos,
    reference_bases=ref,
    alternate_bases=alt,
)

# distance=50 reports splice scores 50 bp on each side of the variant.
result = run_pangolin_score_variants(
    PangolinScoreVariantsInput(variants=[variant]),
    PangolinScoreVariantsConfig(device="cpu", distance=50),
)

effect = result.results[0]
print(f"success: {result.success}")
print(f"largest increase: {effect.increase_score:.4f} at {effect.increase_position} bp")
print(f"largest decrease: {effect.decrease_score:.4f} at {effect.decrease_position} bp")
print(f"metrics: {effect.metrics}")

In [ ]:
# Schemas for the variant-scoring tool: config (tissues, distance, device) and output.
display_api_reference("pangolin-score-variants", "config", "run_pangolin_score_variants")
display_api_reference("pangolin-score-variants", "output", "run_pangolin_score_variants")

In [ ]:
# Export the variant-effect results. The default format is JSON; CSV is also
# supported via file_format="csv" (one row per variant with gain/loss summaries).
from pathlib import Path
Path("example_output").mkdir(exist_ok=True)
result.export(name="pangolin_variants", export_path="./example_output")
print("Exported results to example_output/pangolin_variants.json (default JSON; pass file_format='csv' for CSV)")